# Context-Aware DQN Training on Google Quantum Virtual Machine (Willow)

Training a **Dueling Double DQN** with **Prioritized Experience Replay** for adaptive QKD parameter optimization, using **BB84 quantum circuits** executed on Google's Quantum Virtual Machine with real **Willow processor noise calibration data**.

### Key Differences from Classical Simulation (`quantum_new.ipynb`)
| Aspect | Classical (`quantum_new`) | QVM (this notebook) |
|---|---|---|
| **QBER Source** | Random classical noise | Actual quantum circuit measurement errors |
| **Noise Model** | Gaussian + uniform | Google Willow calibration (gate errors, T1/T2, readout) |
| **Eavesdropper** | +3% QBER constant | Depolarizing channel (intercept-resend physics) |
| **Turbulence** | Uniform random 5-10% | Depolarizing channel on quantum channel |
| **Simulator** | None (pure math) | Cirq + qsim (statevector + noise) |

> **Recommended:** Run in **Google Colab** for best `qsimcirq` compatibility. For local runs, ensure `cirq-google` and `qsimcirq` are installed.

In [4]:
# ==========================================
# 0. INSTALL DEPENDENCIES (run once)
# ==========================================
# Uncomment for Colab / fresh environments:
!pip install cirq-google qsimcirq torch matplotlib numpy --quiet
# For local installs:
!pip install cirq-google qsimcirq

In [2]:
# ==========================================
# 1. IMPORTS + DEVICE CONFIGURATION
# ==========================================
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import random
import sys
import os
import time
import warnings
import matplotlib.pyplot as plt
from collections import deque
from datetime import datetime

warnings.filterwarnings('ignore')

# Quantum imports
import cirq
import cirq_google

try:
    import qsimcirq
    HAS_QSIM = True
    print("qsimcirq loaded — high-performance noisy simulation available")
except ImportError:
    HAS_QSIM = False
    print("qsimcirq not found — falling back to cirq.DensityMatrixSimulator (slower)")

# GPU / CPU detection
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    print(f"CUDA GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB)")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
    print("Apple MPS Backend Detected")
else:
    DEVICE = torch.device('cpu')
    print("Using CPU")
print(f"PyTorch device: {DEVICE}\n")

ModuleNotFoundError: No module named 'cirq'

In [ ]:
# ==========================================
# 2. QUANTUM VIRTUAL MACHINE SETUP (Willow)
# ==========================================
# Choose processor: "willow_pink", "rainbow", or "weber"
PROCESSOR_ID = "willow_pink"

print(f"Loading noise calibration for: {PROCESSOR_ID}")
noise_props = cirq_google.engine.load_device_noise_properties(PROCESSOR_ID)
noise_model = cirq_google.NoiseModelFromGoogleNoiseProperties(noise_props)

# Build noisy simulator
if HAS_QSIM:
    qvm_sim = qsimcirq.QSimSimulator(noise=noise_model)
    print("Using qsim backend (GPU-accelerated if available)")
else:
    qvm_sim = cirq.DensityMatrixSimulator(noise=noise_model)
    print("Using DensityMatrixSimulator backend")

# Build virtual engine
device = cirq_google.engine.create_device_from_processor_id(PROCESSOR_ID)
cal = cirq_google.engine.load_median_device_calibration(PROCESSOR_ID)
sim_processor = cirq_google.engine.SimulatedLocalProcessor(
    processor_id=PROCESSOR_ID,
    sampler=qvm_sim,
    device=device,
    calibrations={cal.timestamp // 1000: cal}
)
sim_engine = cirq_google.engine.SimulatedLocalEngine([sim_processor])
qvm_sampler = sim_engine.get_sampler(PROCESSOR_ID)

# Select qubits for BB84 (single-qubit ops only, no connectivity needed)
available_qubits = sorted(device.metadata.qubit_set)
N_SIGNAL_QUBITS = 20  # qubits per BB84 block circuit
BB84_QUBITS = available_qubits[:N_SIGNAL_QUBITS]
BB84_REPS = 500  # repetitions per circuit (N_QUBITS × REPS ≈ 10,000 raw measurements)

print(f"\nQVM ready: {PROCESSOR_ID}")
print(f"  Signal qubits: {N_SIGNAL_QUBITS}")
print(f"  Repetitions/circuit: {BB84_REPS}")
print(f"  Raw measurements/block: {N_SIGNAL_QUBITS * BB84_REPS:,}")
print(f"  Total device qubits: {len(available_qubits)}")
print(f"\nQubit grid (first {N_SIGNAL_QUBITS}):")
for q in BB84_QUBITS:
    print(f"  {q}", end="")
print()

In [ ]:
# ==========================================
# 3. BB84 QUANTUM CIRCUIT BUILDER
# ==========================================
class BB84QuantumChannel:
    """
    Builds and runs BB84 prepare-and-measure circuits on the QVM.
    
    Protocol per block:
      1. Alice randomly chooses bits and bases (Z or X) for N qubits
      2. Alice prepares quantum states: Z-basis → |0⟩/|1⟩, X-basis → |+⟩/|−⟩
      3. Optional eavesdropper: depolarizing channel (≈ intercept-resend, 25% QBER)
      4. Optional turbulence: additional depolarizing noise on the channel
      5. Bob randomly chooses measurement bases
      6. Bob measures (H before measurement for X-basis)
      7. Sifting: keep only matching-basis measurements
      8. QBER = error rate in sifted key
    
    The QVM's Willow noise model adds realistic gate errors, T1/T2 decoherence,
    and readout errors on top of any channel effects.
    """
    
    def __init__(self, sampler, qubits, repetitions=500):
        self.sampler = sampler
        self.qubits = list(qubits)
        self.n_qubits = len(self.qubits)
        self.repetitions = repetitions
        self._circuit_cache = {}
    
    def run_bb84_block(self, eve_active=False, turbulence_level=0.0):
        """
        Execute one BB84 block on the QVM and return measured QBER.
        
        Args:
            eve_active: If True, insert intercept-resend depolarization (~3% QBER)
            turbulence_level: Depolarizing probability for channel turbulence [0.0 - 0.15]
        
        Returns:
            qber: Quantum Bit Error Rate from sifted key measurements
        """
        n = self.n_qubits
        
        # Alice's random choices
        alice_bits = np.random.randint(0, 2, n)
        alice_bases = np.random.randint(0, 2, n)   # 0=Z, 1=X
        
        # Bob's random choices
        bob_bases = np.random.randint(0, 2, n)     # 0=Z, 1=X
        
        # Build the BB84 circuit
        circuit = cirq.Circuit()
        
        # ── Alice Preparation ──
        # Bit encoding: apply X for bit=1
        alice_x_qubits = [self.qubits[i] for i in range(n) if alice_bits[i] == 1]
        if alice_x_qubits:
            circuit.append(cirq.X.on_each(*alice_x_qubits))
        
        # Basis encoding: apply H for X-basis
        alice_h_qubits = [self.qubits[i] for i in range(n) if alice_bases[i] == 1]
        if alice_h_qubits:
            circuit.append(cirq.H.on_each(*alice_h_qubits))
        
        # ── Channel Effects (Skipped in Circuit) ──
        # Because the QVM strictly checks whether gates belong to the hardware's 
        # supported gateset, we cannot inject arbitrary `cirq.depolarize` channels.
        # Instead, we apply these depolarizing channel effects classically as 
        # post-measurement bit-flips. This is mathematically equivalent for BB84
        # (a depolarizing channel with probability `p` effectively causes a Z-basis
        # or X-basis measurement bit-flip with probability `2p/3`).
        
        # ── Bob Measurement ──
        # Apply H for X-basis measurement
        bob_h_qubits = [self.qubits[i] for i in range(n) if bob_bases[i] == 1]
        if bob_h_qubits:
            circuit.append(cirq.H.on_each(*bob_h_qubits))
        
        # Measure all qubits
        circuit.append(cirq.measure(*self.qubits, key='bb84'))
        
        # ── Execute on QVM ──
        result = self.sampler.run(circuit, repetitions=self.repetitions)
        measurements = result.measurements['bb84'].copy()  # shape: (repetitions, n_qubits)
        
        # ── Apply Channel Effects (Post-processing) ──
        flip_prob = 0.0
        if eve_active:
            flip_prob += 0.045 * (2.0 / 3.0)  # ~3% additional error
        if turbulence_level > 0:
            flip_prob += turbulence_level * (2.0 / 3.0)
            
        if flip_prob > 0:
            # Apply random bit-flips based on the combined depolarization probability
            flips = np.random.rand(*measurements.shape) < flip_prob
            measurements = np.logical_xor(measurements, flips).astype(int)
        
        # ── Sifting: keep only matching-basis results ──
        matching_mask = (alice_bases == bob_bases)
        n_matching = matching_mask.sum()
        
        if n_matching == 0:
            # Extremely rare: all bases mismatched. Return baseline QBER.
            return 0.02
        
        # Alice's bits for matching bases
        alice_sifted = alice_bits[matching_mask]  # shape: (n_matching,)
        
        # Bob's measurements for matching bases across all repetitions
        bob_sifted = measurements[:, matching_mask]  # shape: (repetitions, n_matching)
        
        # QBER = fraction of disagreements across all sifted measurements
        errors = (bob_sifted != alice_sifted[np.newaxis, :])
        qber = float(errors.mean())
        
        return np.clip(qber, 0.001, 0.5)
    
    def benchmark(self, n_blocks=20):
        """Run multiple BB84 blocks to characterize baseline QVM noise."""
        print("Benchmarking QVM BB84 channel...")
        qbers = []
        for i in range(n_blocks):
            q = self.run_bb84_block(eve_active=False, turbulence_level=0.0)
            qbers.append(q)
        
        qbers = np.array(qbers)
        print(f"  Baseline QBER: {qbers.mean()*100:.3f}% ± {qbers.std()*100:.3f}%")
        print(f"  Range: [{qbers.min()*100:.3f}%, {qbers.max()*100:.3f}%]")
        print(f"  (This is from Willow hardware noise alone — no eve/turbulence)")
        return qbers

# Initialize BB84 channel
bb84_channel = BB84QuantumChannel(qvm_sampler, BB84_QUBITS, repetitions=BB84_REPS)

# Run benchmark to characterize baseline noise
baseline_qbers = bb84_channel.benchmark(n_blocks=20)

# Test with eavesdropper
print("\nWith eavesdropper:")
eve_qbers = [bb84_channel.run_bb84_block(eve_active=True) for _ in range(10)]
print(f"  QBER with Eve: {np.mean(eve_qbers)*100:.3f}% ± {np.std(eve_qbers)*100:.3f}%")

# Test with turbulence
print("\nWith turbulence (p=0.08):")
turb_qbers = [bb84_channel.run_bb84_block(turbulence_level=0.08) for _ in range(10)]
print(f"  QBER with turbulence: {np.mean(turb_qbers)*100:.3f}% ± {np.std(turb_qbers)*100:.3f}%")

In [ ]:
# ==========================================
# 4. CONFIGURATION
# ==========================================
NUM_EPISODES = 500        # Reduced significantly for realistic QVM training times
STEPS_PER_EPISODE = 20    # Halved to reduce circuits per episode
BATCH_SIZE = 128          # Smaller batch size for faster individual updates 
GAMMA = 0.9
LEARNING_RATE = 0.0005
MEMORY_SIZE = 50000
TARGET_UPDATE_TAU = 0.005
PER_ALPHA = 0.6
PER_BETA_START = 0.4
PER_BETA_END = 1.0
PER_EPSILON = 1e-5
BLOCK_SIZE = N_SIGNAL_QUBITS * BB84_REPS  # Raw measurements per block

# ==========================================
# 5. MATH HELPERS
# ==========================================
def binary_entropy(p):
    p = np.clip(p, 1e-9, 1 - 1e-9)
    return -p * np.log2(p) - (1 - p) * np.log2(1 - p)

def devetak_winter_rate(qber, f_ec=1.16):
    h2 = binary_entropy(qber)
    rate = 1.0 - h2 - f_ec * h2
    return max(0.0, rate)

# ==========================================
# 6. PRIORITIZED EXPERIENCE REPLAY
# ==========================================
class PrioritizedReplayBuffer:
    def __init__(self, capacity, alpha=0.6):
        self.capacity = capacity
        self.alpha = alpha
        self.buffer = []
        self.priorities = np.zeros(capacity, dtype=np.float64)
        self.pos = 0
        self.size = 0

    def push(self, transition):
        max_prio = self.priorities[:self.size].max() if self.size > 0 else 1.0
        if self.size < self.capacity:
            self.buffer.append(transition)
        else:
            self.buffer[self.pos] = transition
        self.priorities[self.pos] = max_prio
        self.pos = (self.pos + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size, beta=0.4):
        prios = self.priorities[:self.size] ** self.alpha
        probs = prios / prios.sum()
        indices = np.random.choice(self.size, batch_size, p=probs, replace=False)
        samples = [self.buffer[i] for i in indices]
        weights = (self.size * probs[indices]) ** (-beta)
        weights /= weights.max()
        return samples, indices, torch.FloatTensor(weights).to(DEVICE)

    def update_priorities(self, indices, td_errors):
        for idx, td in zip(indices, td_errors):
            self.priorities[idx] = abs(td) + PER_EPSILON

    def __len__(self):
        return self.size

# ==========================================
# 7. QVM-BASED QKD ENVIRONMENT
# ==========================================
class QKDEnvironmentQVM:
    """
    QKD Environment where QBER is measured from actual BB84 quantum circuits
    executed on the Google Quantum Virtual Machine (Willow noise model).
    
    State space (7D) — identical to classical version:
      [0] qber × 2.0
      [1] delta_qber × 10.0  (momentum)
      [2] avg_qber × 2.0     (exponential moving average)
      [3] snr × 0.01
      [4] key_pool / 2000.0
      [5] eve_present (binary)
      [6] was_turbulent (binary)
    
    Action space: 40 actions (4 tag lengths × 10 PA ratios)
    """
    
    def __init__(self, bb84_channel):
        self.channel = bb84_channel
        self.key_pool = 1000
        self.prev_qber = 0.01
        self.avg_qber = 0.01
        self.was_turbulent = False
        self.state = None

    def get_observation(self, qber, snr, eve_present):
        delta_qber = (qber - self.prev_qber) * 10.0
        self.prev_qber = qber
        self.avg_qber = 0.8 * self.avg_qber + 0.2 * qber
        return np.array([
            qber * 2.0,
            delta_qber,
            self.avg_qber * 2.0,
            snr * 0.01,
            self.key_pool / 2000.0,
            float(eve_present),
            float(self.was_turbulent),
        ], dtype=np.float32)

    def reset(self):
        self.prev_qber = 0.01
        self.avg_qber = 0.01
        self.was_turbulent = False
        self.key_pool = 1000
        self.state = self.get_observation(0.01, 50.0, False)
        return self.state

    def step(self, action_idx):
        tag_idx = action_idx // 10
        pa_idx = action_idx % 10

        tag_lengths = [32, 64, 96, 128]
        tag_length = tag_lengths[tag_idx]
        pa_ratio = (pa_idx + 1) / 10.0

        # Stochastic channel conditions
        is_turbulent = random.random() < 0.15
        turbulence_level = random.uniform(0.03, 0.08) if is_turbulent else 0.0
        eve_present = random.random() < 0.15

        # ═══════════════════════════════════════════
        # RUN BB84 ON QUANTUM VIRTUAL MACHINE
        # QBER comes from actual quantum measurements!
        # ═══════════════════════════════════════════
        current_qber = self.channel.run_bb84_block(
            eve_active=eve_present,
            turbulence_level=turbulence_level
        )
        current_qber = np.clip(current_qber, 0.001, 0.5)

        snr = 1.0 / (current_qber + 1e-6)
        secure_rate = devetak_winter_rate(current_qber)

        reward = 0.0
        final_keys = 0
        is_secure = True
        done = False

        if self.key_pool < tag_length:
            reward = -5.0
            self.key_pool = 1000
        else:
            self.key_pool -= tag_length
            is_secure = (pa_ratio <= secure_rate)

            if not is_secure:
                overshoot = pa_ratio - secure_rate
                reward = -5.0 - (overshoot * 10.0)
            elif secure_rate < 0.01:
                reward = 1.0 if pa_ratio <= 0.1 else -1.0
            else:
                raw_yield = BLOCK_SIZE * pa_ratio * (1 - current_qber)
                final_keys = int(raw_yield)
                self.key_pool += final_keys
                reward = (secure_rate * final_keys) / 500.0

                if not is_turbulent and pa_ratio > 0.6:
                    reward *= 1.3
                if is_turbulent and is_secure:
                    reward += 3.0

        if self.key_pool > 10000:
            self.key_pool = 10000

        self.was_turbulent = is_turbulent
        self.state = self.get_observation(current_qber, snr, eve_present)

        info = {
            'qber': current_qber,
            'keys': final_keys,
            'auth_cost': tag_length,
            'secure': is_secure,
            'turbulent': is_turbulent,
            'secure_rate': secure_rate,
            'pa_ratio': pa_ratio,
            'eve_present': eve_present,
        }

        return self.state, reward, done, info

# ==========================================
# 8. DUELING DQN AGENT
# ==========================================
class DuelingDQNAgent(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(DuelingDQNAgent, self).__init__()
        self.feature = nn.Sequential(
            nn.Linear(state_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
        )
        self.value_stream = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )
        self.advantage_stream = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim),
        )

    def forward(self, x):
        features = self.feature(x)
        value = self.value_stream(features)
        advantage = self.advantage_stream(features)
        return value + advantage - advantage.mean(dim=1, keepdim=True)

# ==========================================
# 9. TRAINING ENGINE
# ==========================================
def run_qvm_training():
    """Train Context-Aware DQN with BB84 circuits on Google QVM."""
    env = QKDEnvironmentQVM(bb84_channel)
    STATE_DIM = 7
    ACTION_DIM = 40

    policy_net = DuelingDQNAgent(STATE_DIM, ACTION_DIM).to(DEVICE)
    target_net = DuelingDQNAgent(STATE_DIM, ACTION_DIM).to(DEVICE)
    target_net.load_state_dict(policy_net.state_dict())
    target_net.eval()

    optimizer = optim.Adam(policy_net.parameters(), lr=LEARNING_RATE)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPISODES, eta_min=1e-5)
    criterion = nn.SmoothL1Loss(reduction='none')

    replay_buffer = PrioritizedReplayBuffer(MEMORY_SIZE, alpha=PER_ALPHA)
    epsilon = 1.0

    episode_rewards = []
    episode_losses = []
    episode_keys = []
    episode_breaches = []
    episode_qbers = []

    step_count = 0
    total_circuits = 0
    train_start = time.time()

    print(f"{'='*60}")
    print(f"  QVM TRAINING: {NUM_EPISODES} Episodes on {DEVICE}")
    print(f"  Architecture: Dueling Double DQN + PER + Soft Updates")
    print(f"  Batch Size: {BATCH_SIZE}")
    print(f"  BB84 Circuits/step: 1 ({N_SIGNAL_QUBITS} qubits × {BB84_REPS} reps)")
    print(f"  Estimated circuits: {NUM_EPISODES * STEPS_PER_EPISODE:,}")
    print(f"{'='*60}")

    for episode in range(1, NUM_EPISODES + 1):
        state = env.reset()
        ep_reward = 0
        ep_loss = 0
        ep_keys = 0
        ep_breaches = 0
        ep_qbers = []
        loss_count = 0

        beta = PER_BETA_START + (PER_BETA_END - PER_BETA_START) * (episode / NUM_EPISODES)

        for _ in range(STEPS_PER_EPISODE):
            step_count += 1
            total_circuits += 1

            # Epsilon-greedy action selection
            if random.random() < epsilon:
                action = random.randint(0, ACTION_DIM - 1)
            else:
                with torch.no_grad():
                    state_t = torch.FloatTensor(state).unsqueeze(0).to(DEVICE)
                    q_vals = policy_net(state_t)
                    action = torch.argmax(q_vals).item()

            next_state, reward, done, info = env.step(action)
            replay_buffer.push((state, action, reward, next_state, done))

            state = next_state
            ep_reward += reward
            ep_keys += info['keys']
            ep_qbers.append(info['qber'])
            if not info['secure'] and info['keys'] > 0:
                ep_breaches += 1

            # Train on minibatch
            if len(replay_buffer) > BATCH_SIZE:
                samples, indices, is_weights = replay_buffer.sample(BATCH_SIZE, beta)
                b_s, b_a, b_r, b_ns, b_d = zip(*samples)

                s_tensor = torch.FloatTensor(np.stack(b_s)).to(DEVICE)
                a_tensor = torch.LongTensor(b_a).unsqueeze(1).to(DEVICE)
                r_tensor = torch.FloatTensor(b_r).unsqueeze(1).to(DEVICE)
                ns_tensor = torch.FloatTensor(np.stack(b_ns)).to(DEVICE)
                d_tensor = torch.FloatTensor(b_d).unsqueeze(1).to(DEVICE)

                curr_q = policy_net(s_tensor).gather(1, a_tensor)

                with torch.no_grad():
                    best_actions = policy_net(ns_tensor).argmax(1, keepdim=True)
                    next_q = target_net(ns_tensor).gather(1, best_actions)
                    target_q = r_tensor + (GAMMA * next_q * (1 - d_tensor))

                td_errors = (curr_q - target_q).detach().cpu().squeeze().numpy()
                replay_buffer.update_priorities(indices, td_errors)

                element_loss = criterion(curr_q, target_q).squeeze()
                loss = (element_loss * is_weights).mean()

                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(policy_net.parameters(), 1.0)
                optimizer.step()

                ep_loss += loss.item()
                loss_count += 1

                # Soft target update (Polyak averaging)
                for target_param, policy_param in zip(target_net.parameters(), policy_net.parameters()):
                    target_param.data.copy_(
                        TARGET_UPDATE_TAU * policy_param.data + (1.0 - TARGET_UPDATE_TAU) * target_param.data
                    )

            if done:
                break

        epsilon = max(0.05, epsilon * 0.998)
        scheduler.step()

        episode_rewards.append(ep_reward)
        episode_losses.append(ep_loss / max(loss_count, 1))
        episode_keys.append(ep_keys)
        episode_breaches.append(ep_breaches)
        episode_qbers.append(np.mean(ep_qbers))

        if episode % 1 == 0 or episode == NUM_EPISODES:
            progress = (episode / NUM_EPISODES) * 100
            bar_len = 30
            block = int(round(bar_len * progress / 100))
            bar = '#' * block + '-' * (bar_len - block)
            elapsed = time.time() - train_start
            eps_per_sec = episode / max(elapsed, 1e-8)
            avg_rw = np.mean(episode_rewards[-100:])
            avg_qber = np.mean(episode_qbers[-100:]) * 100
            sys.stdout.write(
                f"\r[{bar}] {progress:.1f}% | Ep: {episode} | "
                f"ε: {epsilon:.3f} | Rw: {avg_rw:.1f} | "
                f"QBER: {avg_qber:.2f}% | {eps_per_sec:.2f} ep/s"
            )
            sys.stdout.flush()

    elapsed = time.time() - train_start
    print(f"\n\n{'='*60}")
    print(f"QVM Training Complete in {elapsed:.1f}s ({elapsed / 60:.1f} min)")
    print(f"Total BB84 circuits executed: {total_circuits:,}")
    print(f"Throughput: {NUM_EPISODES / max(elapsed, 1e-8):.2f} episodes/sec")
    print(f"{'='*60}\n")

    # ==========================================
    # 10. EVALUATION
    # ==========================================
    print("Running evaluation on QVM...")
    eval_episodes = 300
    eval_steps = 40

    # RL Agent evaluation
    rl_stats = {'total_keys': 0, 'failures': 0, 'breaches': 0}
    policy_net.eval()
    eval_env = QKDEnvironmentQVM(bb84_channel)

    for ep in range(eval_episodes):
        state = eval_env.reset()
        for _ in range(eval_steps):
            with torch.no_grad():
                state_t = torch.FloatTensor(state).unsqueeze(0).to(DEVICE)
                q_vals = policy_net(state_t)
                action = torch.argmax(q_vals).item()

            state, _, done, info = eval_env.step(action)
            rl_stats['total_keys'] += info['keys']
            if not info['secure'] and info['keys'] > 0:
                rl_stats['breaches'] += 1
            if info['keys'] == 0:
                rl_stats['failures'] += 1
            if done:
                break
        if (ep + 1) % 50 == 0:
            sys.stdout.write(f"\r  RL eval: {ep+1}/{eval_episodes}")
            sys.stdout.flush()

    # Static baseline evaluation
    print(f"\r  RL eval: {eval_episodes}/{eval_episodes} ✓")
    static_stats = {'total_keys': 0, 'failures': 0, 'breaches': 0}
    static_env = QKDEnvironmentQVM(bb84_channel)
    static_env.reset()

    for step in range(eval_episodes * eval_steps):
        _, _, done, info = static_env.step(14)  # Fixed: tag=64, pa=0.5
        static_stats['total_keys'] += info['keys']
        if not info['secure'] and info['keys'] > 0:
            static_stats['breaches'] += 1
        if info['keys'] == 0:
            static_stats['failures'] += 1
        if done:
            static_env.reset()
        if (step + 1) % 1000 == 0:
            sys.stdout.write(f"\r  Static eval: {step+1}/{eval_episodes*eval_steps}")
            sys.stdout.flush()

    print(f"\r  Static eval: {eval_episodes*eval_steps}/{eval_episodes*eval_steps} ✓")

    print(f"\n{'='*65}")
    print(f"{'QVM BENCHMARK (BB84 on Willow Noise Model)':^65}")
    print(f"{'='*65}")
    print(f"{'Metric':<25} | {'Static (Fixed)':<18} | {'AI Agent (QVM)':<18}")
    print(f"{'-'*65}")
    print(f"{'Total Secure Keys':<25} | {static_stats['total_keys']:<18,} | {rl_stats['total_keys']:<18,}")
    print(f"{'Security Breaches':<25} | {static_stats['breaches']:<18} | {rl_stats['breaches']:<18}")
    print(f"{'Failed/Aborted Blocks':<25} | {static_stats['failures']:<18} | {rl_stats['failures']:<18}")
    print(f"{'='*65}")

    if static_stats['total_keys'] > 0:
        improvement = ((rl_stats['total_keys'] - static_stats['total_keys']) / static_stats['total_keys']) * 100
        print(f"\nKey Yield Improvement: {improvement:+.2f}%")
    if static_stats['failures'] > 0:
        fail_reduction = ((static_stats['failures'] - rl_stats['failures']) / static_stats['failures']) * 100
        print(f"Failure Rate Reduction: {fail_reduction:+.2f}%")

    if rl_stats['total_keys'] > static_stats['total_keys']:
        print("\n>>> SUCCESS: AI outperforms Static Baseline on QVM.")
    else:
        print("\n>>> NOTE: AI still converging — try increasing episodes.")

    return (episode_rewards, episode_losses, episode_keys, episode_breaches,
            episode_qbers, static_stats, rl_stats, policy_net)

# ==========================================
# 11. RUN
# ==========================================
if __name__ == '__main__':
    (rewards, losses, keys, breaches, qbers,
     static_stats, rl_stats, trained_model) = run_qvm_training()

In [ ]:
# ==========================================
# 12. TRAINING VISUALIZATIONS
# ==========================================
OUTPUT_DIR = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

def smooth(data, window=50):
    """Simple moving average for smoother plots."""
    if len(data) < window:
        return data
    return np.convolve(data, np.ones(window)/window, mode='valid')

fig, axes = plt.subplots(3, 2, figsize=(15, 14))
fig.suptitle("Context-Aware Dueling Double DQN — QVM Training Analysis\n(BB84 on Google Willow Noise Model)",
             fontsize=14, fontweight='bold')

# (a) Episode Reward
ax = axes[0, 0]
ax.plot(smooth(rewards, 50), color='#2196F3', linewidth=0.8)
ax.set_title("(a) Episode Reward (50-ep Moving Avg)")
ax.set_xlabel("Episode")
ax.set_ylabel("Cumulative Reward")
ax.grid(True, alpha=0.3)

# (b) Training Loss
ax = axes[0, 1]
ax.plot(smooth(losses, 50), color='#F44336', linewidth=0.8)
ax.set_title("(b) Training Loss (Huber)")
ax.set_xlabel("Episode")
ax.set_ylabel("Loss")
ax.grid(True, alpha=0.3)

# (c) Secure Keys per Episode
ax = axes[1, 0]
ax.plot(smooth(keys, 100), color='#4CAF50', linewidth=0.8)
ax.set_title("(c) Secure Key Yield per Episode")
ax.set_xlabel("Episode")
ax.set_ylabel("Keys (100-ep avg)")
ax.grid(True, alpha=0.3)

# (d) QBER from QVM (NEW — unique to QVM training)
ax = axes[1, 1]
ax.plot(smooth(qbers, 50), color='#9C27B0', linewidth=0.8, label='Avg QBER')
ax.axhline(y=0.11, color='red', linestyle='--', alpha=0.5, label='BB84 Threshold (11%)')
ax.set_title("(d) QVM-Measured QBER per Episode")
ax.set_xlabel("Episode")
ax.set_ylabel("QBER")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# (e) Breaches per Episode
ax = axes[2, 0]
ax.plot(smooth(breaches, 50), color='#FF5722', linewidth=0.8)
ax.set_title("(e) Security Breaches per Episode")
ax.set_xlabel("Episode")
ax.set_ylabel("Breaches (50-ep avg)")
ax.grid(True, alpha=0.3)

# (f) Comparative Bar Chart
ax = axes[2, 1]
metrics = ['Secure Keys\n(×10³)', 'Failures', 'Breaches']
static_vals = [static_stats['total_keys']/1e3, static_stats['failures'], static_stats['breaches']]
rl_vals = [rl_stats['total_keys']/1e3, rl_stats['failures'], rl_stats['breaches']]

x = np.arange(len(metrics))
width = 0.35
ax.bar(x - width/2, static_vals, width, label='Static Protocol', color='#FF9800', alpha=0.85)
ax.bar(x + width/2, rl_vals, width, label='Dueling DDQN (QVM)', color='#2196F3', alpha=0.85)
ax.set_title("(f) QVM Evaluation Comparison")
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
save_path = os.path.join(OUTPUT_DIR, 'qvm_qkd_training.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Figure saved to {save_path}")

In [ ]:
# ==========================================
# 13. SAVE QVM-TRAINED MODEL
# ==========================================
MODELS_DIR = os.path.join(OUTPUT_DIR, 'models')
os.makedirs(MODELS_DIR, exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
checkpoint_path = os.path.join(MODELS_DIR, f'dueling_ddqn_qkd_qvm_{timestamp}.pth')

checkpoint = {
    # ── Model Architecture ──
    'model_state_dict': agent.online_net.state_dict(),
    'target_state_dict': agent.target_net.state_dict(),
    'optimizer_state_dict': agent.optimizer.state_dict(),
    'scheduler_state_dict': agent.scheduler.state_dict(),

    # ── Training Config ──
    'config': {
        'state_dim': CFG_STATE_DIM,
        'action_dim': CFG_ACTION_DIM,
        'hidden': 256,
        'episodes': CFG_EPISODES,
        'steps_per_episode': CFG_STEPS,
        'gamma': CFG_GAMMA,
        'lr': CFG_LR,
        'batch_size': CFG_BATCH,
        'buffer_size': CFG_BUFFER,
        'tau': CFG_TAU,
    },

    # ── QVM Metadata ──
    'qvm': {
        'backend': 'Google Cirq QVM (qsimcirq)',
        'processor_id': PROCESSOR_ID,
        'n_signal_qubits': N_SIGNAL_QUBITS,
        'repetitions': REPETITIONS,
        'noise_model': 'NoiseModelFromGoogleNoiseProperties (Willow)',
        'eve_depolarize': 0.045,
        'turbulence_depolarize_range': [0.03, 0.08],
    },

    # ── Training Results ──
    'training': {
        'total_episodes': len(rewards),
        'final_reward_avg': float(np.mean(rewards[-100:])),
        'final_loss_avg': float(np.mean(losses[-100:])) if losses else 0.0,
        'final_qber_avg': float(np.mean(qbers[-100:])),
        'final_keys_avg': float(np.mean(keys[-100:])),
    },

    # ── Evaluation Results ──
    'evaluation': {
        'static_protocol': static_stats,
        'rl_agent': rl_stats,
    },

    # ── Device Info ──
    'device': str(DEVICE),
    'timestamp': timestamp,
}

torch.save(checkpoint, checkpoint_path)
print(f"QVM-trained model saved to:\n  {checkpoint_path}")
print(f"  File size: {os.path.getsize(checkpoint_path) / 1e6:.2f} MB")
print(f"\nCheckpoint keys: {list(checkpoint.keys())}")
print(f"QVM config: {checkpoint['qvm']}")
print(f"\nTraining summary:")
print(f"  Episodes:       {checkpoint['training']['total_episodes']}")
print(f"  Final Reward:   {checkpoint['training']['final_reward_avg']:.2f}")
print(f"  Final QBER:     {checkpoint['training']['final_qber_avg']:.4f}")
print(f"  Final Keys:     {checkpoint['training']['final_keys_avg']:.1f}")
print(f"\n Model ready for deployment in qkd_backend.py")